# NeTwatch
## Agentic AI for Intelligent Network Support

**Problem Statement**
Customers with internet connectivity problems often need account verification, outage checking, diagnosis, multiple troubleshooting steps, and escalation.

**Solution**
NeTwatch is a lightweight, explainable agentic AI prototype that:
- understands natural-language customer input using supervised NLP
- maintains an explicit agent state
- selects the next action dynamically
- calls simulated support tools
- uses historical case memory with TF-IDF similarity
- incorporates customer feedback as observations
- resolves the case when confirmed, or escalates when needed

**Prototype stack:** Python, Google Colab, scikit-learn, TF-IDF, Logistic Regression.

> No external LLM/API is required. The goal is a transparent first-month Agentic AI prototype rather than an over-engineered production system.


In [ ]:
# CELL 1 - IMPORTS & CONFIGURATION
import re
import random
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

MAX_TROUBLESHOOTING_ATTEMPTS = 4
MEMORY_SIMILARITY_THRESHOLD = 0.12

print("NeTwatch environment initialized successfully.")


In [ ]:
# CELL 2 - CUSTOMER KNOWLEDGE BASE
customers = {
    "C1001": {
        "name": "Rahul Sharma", "plan": "Fiber 100 Mbps",
        "account_status": "Active", "area": "Pune",
        "router_status": "Online", "internet_status": "Disconnected"
    },
    "C1002": {
        "name": "Priya Patil", "plan": "Fiber 200 Mbps",
        "account_status": "Active", "area": "Pune",
        "router_status": "Offline", "internet_status": "Disconnected"
    },
    "C1003": {
        "name": "Amit Kulkarni", "plan": "Fiber 100 Mbps",
        "account_status": "Suspended", "area": "Mumbai",
        "router_status": "Online", "internet_status": "Disconnected"
    },
    "C1004": {
        "name": "Sneha Joshi", "plan": "Fiber 500 Mbps",
        "account_status": "Active", "area": "Mumbai",
        "router_status": "Online", "internet_status": "Disconnected"
    }
}
print(f"Customer knowledge base loaded: {len(customers)} customers")


In [ ]:
# CELL 3 - NETWORK / OUTAGE KNOWLEDGE BASE
network_status = {
    "Pune": {
        "outage": False, "severity": "None",
        "message": "No active network outage reported."
    },
    "Mumbai": {
        "outage": True, "severity": "Major",
        "message": "A network outage is currently affecting the area."
    }
}

print("Network knowledge base loaded.")


In [ ]:
# CELL 4 - SUPPORT TOOLS: ACCOUNT, OUTAGE, DIAGNOSIS
def retrieve_account(customer_id):
    if customer_id not in customers:
        return {"success": False, "message": "Customer ID not found."}
    return {"success": True, "account": customers[customer_id].copy()}


def check_outage(area):
    if area not in network_status:
        return {"success": False, "outage": None, "message": "Area information unavailable."}
    status = network_status[area]
    return {"success": True, "outage": status["outage"],
            "severity": status["severity"], "message": status["message"]}


def diagnose_connection(account, outage_info, state=None):
    if account["account_status"] != "Active":
        return {"diagnosis": "Account issue", "reason": "The customer account is not active."}
    if outage_info["outage"]:
        return {"diagnosis": "Area network outage", "reason": outage_info["message"]}
    if account["router_status"] == "Offline":
        return {"diagnosis": "Router connectivity issue", "reason": "The router is currently offline."}
    if account["router_status"] == "Online":
        return {
            "diagnosis": "Possible WAN, DNS or device connectivity issue",
            "reason": "The router is online and no area outage is reported. Further troubleshooting is required."
        }
    return {"diagnosis": "Unknown connectivity issue", "reason": "Insufficient information for a specific diagnosis."}

print("Account, outage and diagnosis tools loaded.")


In [ ]:
# CELL 5 - CASE MEMORY TOOL
case_memory = [
    {"case_id": "CASE001", "problem": "Internet not working and router is offline",
     "resolution": "Check power connection and restart the router."},
    {"case_id": "CASE002", "problem": "Router online but internet not working",
     "resolution": "Check WAN cable, DNS and device connectivity."},
    {"case_id": "CASE003", "problem": "Internet unavailable during area outage",
     "resolution": "Inform the customer about the outage and escalate if required."},
    {"case_id": "CASE004", "problem": "Internet not working after router restart",
     "resolution": "Check WAN connection and perform advanced troubleshooting."},
    {"case_id": "CASE005", "problem": "WAN light normal but internet unavailable on multiple devices",
     "resolution": "Perform service-side diagnostics and escalate if unresolved."}
]

memory_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
memory_matrix = memory_vectorizer.fit_transform([c["problem"] for c in case_memory])

def find_similar_cases(problem, top_k=2):
    query_vector = memory_vectorizer.transform([problem])
    similarities = cosine_similarity(query_vector, memory_matrix).flatten()
    ranked = similarities.argsort()[::-1][:top_k]
    return [{
        "case_id": case_memory[i]["case_id"],
        "problem": case_memory[i]["problem"],
        "resolution": case_memory[i]["resolution"],
        "similarity": round(float(similarities[i]), 3)
    } for i in ranked]

print(f"Case memory loaded: {len(case_memory)} historical cases")


In [ ]:
# CELL 6 - INTENT TRAINING DATA
training_data = [
    ("my internet is not working", "internet_down"),
    ("internet stopped working", "internet_down"),
    ("I have no internet", "internet_down"),
    ("my connection is down", "internet_down"),
    ("wifi is not working", "internet_down"),
    ("I cannot access the internet", "internet_down"),
    ("the router is powered on", "router_powered"),
    ("router is on", "router_powered"),
    ("my router has power", "router_powered"),
    ("the router lights are on", "router_powered"),
    ("the internet light is red", "red_wan_light"),
    ("WAN light is red", "red_wan_light"),
    ("my router has a red internet light", "red_wan_light"),
    ("the WAN indicator is red", "red_wan_light"),
    ("I restarted the router", "restart_completed"),
    ("router has been restarted", "restart_completed"),
    ("I already rebooted it", "restart_completed"),
    ("I rebooted my router", "restart_completed"),
    ("it is working now", "issue_resolved"),
    ("internet is back", "issue_resolved"),
    ("the problem is fixed", "issue_resolved"),
    ("everything is working", "issue_resolved"),
    ("it is still not working", "issue_unresolved"),
    ("the problem is still there", "issue_unresolved"),
    ("internet is still down", "issue_unresolved"),
    ("it did not fix the problem", "issue_unresolved"),
    ("still no internet", "issue_unresolved")
]

texts = [x[0] for x in training_data]
labels = [x[1] for x in training_data]
print("Training examples:", len(training_data))
print("Intent categories:", sorted(set(labels)))


In [ ]:
# CELL 7 - TRAIN THE SUPERVISED NLP MODEL
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
X = vectorizer.fit_transform(texts)
intent_model = LogisticRegression(max_iter=1000, random_state=42)
intent_model.fit(X, labels)

def analyze_user_input(text):
    text_vector = vectorizer.transform([text])
    predicted_intent = intent_model.predict(text_vector)[0]
    probabilities = intent_model.predict_proba(text_vector)[0]
    confidence = float(max(probabilities))
    customer_match = re.search(r"\bC\d{4}\b", text.upper())
    return {
        "intent": predicted_intent,
        "confidence": round(confidence, 3),
        "customer_id": customer_match.group() if customer_match else None
    }

print("Supervised intent classifier trained successfully.")
print("Features learned:", X.shape[1])


In [ ]:
# CELL 8 - AGENT STATE
def create_agent_state():
    return {
        "customer_id": None, "customer": None, "issue": None,
        "intent": None, "confidence": 0.0, "plan": None,
        "account_status": None, "area": None, "router_status": None,
        "internet_status": None, "outage": None, "diagnosis": None,
        "similar_cases": [], "memory_checked": False,
        "troubleshooting_attempts": 0, "steps_attempted": [],
        "feedback_history": [], "restart_completed": False,
        "wan_checked": False, "wan_light_normal": None,
        "multi_device_tested": False, "resolved": False,
        "escalated": False, "ticket_id": None,
        "conversation": []
    }

print("Agent state schema ready.")


In [ ]:
# CELL 9 - FEEDBACK INTERPRETATION
def interpret_feedback(text, last_step=None):
    t = text.lower().strip()

    # Strong negative phrases are checked first because they may contain words like "work".
    negative = [
        "still not working", "still doesn't work", "doesn't work",
        "does not work", "not working", "didn't work", "did not work",
        "not fixed", "no", "nah", "nope", "nothing changed"
    ]
    positive = [
        "yes", "y", "yep", "yeah", "works", "working now",
        "it works", "fixed", "resolved", "internet is back",
        "everything is working", "all good"
    ]
    already_done = [
        "already restarted", "already rebooted", "restarted already",
        "i rebooted it", "i restarted it"
    ]
    router_online = [
        "router is on", "router is online", "router lights are on",
        "it is on", "power is on"
    ]
    wan_normal = [
        "light is normal", "wan light is normal", "internet light is normal",
        "indicator is normal", "light is fine"
    ]

    if any(p in t for p in already_done):
        return "restart_completed"
    if any(p in t for p in wan_normal):
        return "wan_light_normal"
    if any(p in t for p in negative):
        return "issue_unresolved"
    if any(p in t for p in positive):
        return "issue_resolved"
    if any(p in t for p in router_online):
        return "router_online"
    return "unknown"

print("Context-aware feedback interpreter ready.")


In [ ]:
# CELL 10 - TROUBLESHOOTING TOOL
def troubleshoot_connection(state):
    attempts = state["troubleshooting_attempts"]
    router = state["router_status"]

    if router == "Offline":
        steps = [
            "Please check whether the router is powered on and connected to electricity.",
            "Please restart the router and wait about 60 seconds.",
            "Please check the router's Internet/WAN indicator light.",
            "Please check the power adapter and try a different power outlet if available."
        ]
    else:
        steps = [
            "Your router is online. Please check that the WAN/Internet cable is securely connected.",
            "Please check whether the Internet/WAN indicator light is normal.",
            "Please test the connection on another device to determine whether the issue is device-specific.",
            "Please restart the affected device and check whether DNS/network access returns."
        ]

    index = min(attempts, len(steps) - 1)
    step = steps[index]
    state["troubleshooting_attempts"] += 1
    state["steps_attempted"].append(step)
    return {"success": True, "step": step, "attempt": state["troubleshooting_attempts"]}


def escalate_issue(state, reason):
    ticket_id = f"NT-{random.randint(1000, 9999)}"
    state["escalated"] = True
    state["ticket_id"] = ticket_id
    return {
        "ticket_id": ticket_id,
        "customer_id": state["customer_id"],
        "issue": state["issue"],
        "diagnosis": state["diagnosis"],
        "reason": reason,
        "attempts": state["troubleshooting_attempts"],
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

print("Troubleshooting and escalation tools loaded.")


In [ ]:
# CELL 11 - AGENT DECISION ENGINE
def decide_next_action(state):
    # The agent checks state in priority order and selects ONE next action.
    if state["customer_id"] is None:
        return "ask_customer_id"
    if state["customer"] is None:
        return "retrieve_account"
    if state["account_status"] != "Active":
        return "escalate"
    if state["outage"] is None:
        return "check_outage"
    if state["outage"] is True:
        return "inform_outage"
    if state["diagnosis"] is None:
        return "diagnose"
    if state["resolved"]:
        return "resolve_case"
    if not state["memory_checked"]:
        return "search_memory"
    if state["troubleshooting_attempts"] >= MAX_TROUBLESHOOTING_ATTEMPTS:
        return "escalate"
    return "troubleshoot"


def action_reason(action, state):
    reasons = {
        "ask_customer_id": "Account information is required before support actions can be performed.",
        "retrieve_account": "The customer ID is known, so the account tool can provide service and router context.",
        "check_outage": "The agent must distinguish an area-wide outage from an individual connection problem.",
        "diagnose": "Account and outage observations are available, so the agent can classify the likely issue.",
        "search_memory": "Historical cases may provide a useful troubleshooting path for the current diagnosis.",
        "troubleshoot": "The issue is unresolved and the agent has remaining troubleshooting attempts.",
        "inform_outage": "An active area outage explains the connectivity problem.",
        "resolve_case": "The customer has confirmed that the issue is resolved.",
        "escalate": "Automated troubleshooting cannot safely resolve the issue within the available steps."
    }
    return reasons.get(action, "The action was selected from the current agent state.")

print("Agent decision engine ready.")


In [ ]:
# CELL 12 - TOOL EXECUTOR
def execute_action(action, state):
    if action == "retrieve_account":
        result = retrieve_account(state["customer_id"])
        if not result["success"]:
            return result["message"]
        account = result["account"]
        state["customer"] = account
        state["plan"] = account["plan"]
        state["account_status"] = account["account_status"]
        state["area"] = account["area"]
        state["router_status"] = account["router_status"]
        state["internet_status"] = account["internet_status"]
        return f"Account retrieved for {account['name']} ({account['plan']})."

    if action == "check_outage":
        result = check_outage(state["area"])
        if result["success"]:
            state["outage"] = result["outage"]
        return result["message"]

    if action == "diagnose":
        outage_info = check_outage(state["area"])
        result = diagnose_connection(state["customer"], outage_info, state)
        state["diagnosis"] = result["diagnosis"]
        return f"Diagnosis: {result['diagnosis']}. {result['reason']}"

    if action == "search_memory":
        query = state["issue"] or "internet connectivity problem"
        if state["diagnosis"]:
            query += " " + state["diagnosis"]
        results = find_similar_cases(query)
        strong = [r for r in results if r["similarity"] >= MEMORY_SIMILARITY_THRESHOLD]
        state["similar_cases"] = strong
        state["memory_checked"] = True
        if strong:
            best = strong[0]
            return (f"Historical case {best['case_id']} found "
                    f"(similarity {best['similarity']:.2f}). "
                    f"Previous resolution: {best['resolution']}")
        return "No strong historical case match found. Continuing with current-state troubleshooting."

    if action == "troubleshoot":
        return troubleshoot_connection(state)["step"]

    if action == "inform_outage":
        return "An active network outage is affecting your area. The network team needs to handle the service-side issue."

    if action == "escalate":
        ticket = escalate_issue(state, "Automated troubleshooting could not resolve the issue.")
        return f"Escalating to technical support. Ticket created: {ticket['ticket_id']}. Diagnostic history will be attached."

    if action == "resolve_case":
        state["resolved"] = True
        return "The issue has been marked as resolved."

    return "No valid action selected."

print("Agent tool executor ready.")


In [ ]:
# CELL 13 - DYNAMIC NETWATCH AGENT
def print_final_state(state):
    print("\n" + "=" * 60)
    print("NeTwatch - FINAL AGENT STATE")
    print("=" * 60)
    fields = [
        ("Customer ID", state["customer_id"]),
        ("Customer", state["customer"]["name"] if state["customer"] else None),
        ("Intent", state["intent"]),
        ("Account Status", state["account_status"]),
        ("Router Status", state["router_status"]),
        ("Area", state["area"]),
        ("Outage", state["outage"]),
        ("Diagnosis", state["diagnosis"]),
        ("Troubleshooting Attempts", state["troubleshooting_attempts"]),
        ("Resolved", state["resolved"]),
        ("Escalated", state["escalated"]),
        ("Ticket ID", state["ticket_id"])
    ]
    for key, value in fields:
        print(f"{key:<26}: {value}")
    print("=" * 60)


def run_netwatch():
    state = create_agent_state()
    print("=" * 60)
    print("NeTwatch - Network Support Agent")
    print("=" * 60)
    print("Type 'exit' at any time to end the session.\n")

    user_input = input("Customer: ").strip()
    if user_input.lower() == "exit":
        return state

    state["conversation"].append({"role": "customer", "message": user_input})
    analysis = analyze_user_input(user_input)
    state["intent"] = analysis["intent"]
    state["confidence"] = analysis["confidence"]
    state["issue"] = user_input
    if analysis["customer_id"]:
        state["customer_id"] = analysis["customer_id"]

    print(f"\n[NeTwatch NLP] Intent: {state['intent']} | Confidence: {state['confidence']:.3f}")

    for step_number in range(12):
        action = decide_next_action(state)
        print(f"\n[Agent Step {step_number + 1}] Selected action: {action}")
        print(f"[Agent Reason] {action_reason(action, state)}")

        if action == "ask_customer_id":
            response = input("NeTwatch: Please provide your customer ID: ").strip()
            if response.lower() == "exit":
                break
            state["customer_id"] = response.upper()
            state["conversation"].append({"role": "customer", "message": response})
            continue

        observation = execute_action(action, state)
        print(f"NeTwatch: {observation}")

        if action in {"inform_outage", "escalate", "resolve_case"}:
            break

        if action == "troubleshoot":
            response = input("Customer: Did this resolve the issue? ").strip()
            if response.lower() == "exit":
                break
            state["conversation"].append({"role": "customer", "message": response})
            state["feedback_history"].append(response)
            feedback = interpret_feedback(response, state["steps_attempted"][-1])
            print(f"[Feedback] Interpreted as: {feedback}")

            if feedback == "issue_resolved":
                state["resolved"] = True
                state["internet_status"] = "Connected"
            elif feedback == "restart_completed":
                state["restart_completed"] = True
                # A customer statement is an observation, so update the agent state.
                state["router_status"] = "Online"
            elif feedback == "wan_light_normal":
                state["wan_light_normal"] = True
                state["wan_checked"] = True
            elif feedback == "router_online":
                state["router_status"] = "Online"
            elif feedback == "issue_unresolved":
                state["resolved"] = False
            else:
                print("[Feedback] Ambiguous response. The agent will continue from the existing state.")

    print_final_state(state)
    return state

print("Dynamic NeTwatch agent ready.")


In [ ]:
# CELL 14 - LAUNCH / DEMONSTRATION
print("NeTwatch is ready for the jury demonstration.")
print("Run the next cell to start the interactive agent.")


In [ ]:
# CELL 15 - START INTERACTIVE AGENT
final_state = run_netwatch()
